# Supplementary figuresFig. S10 — the field-controlled redistribution reproduced across sixhit-intensity windows: below the warm dense matter transition, across it, andabove it.Fig. S11 — field dependence of the full ion time-of-flight spectrum, with thefield-insensitive Ar3+ reference and the silica-framework ions.

> **Data.** These notebooks read the preprocessed per-run HDF5 files produced by> `scripts/preprocess.py`. Point `EUXFEL_NANO_DATA` at the directory holding> them. The raw data is at <https://doi.org/10.22003/XFEL.EU-DATA-003408-00>,> under embargo until publication.>> Figures 1-3 can be rebuilt **without** this data from the arrays already in> `figures/data/` — see `figures/README.md`.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import xarray as xr
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import gridspec, cm
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
from scipy import constants
from scipy.optimize import curve_fit

from euxfel_nano import config
from euxfel_nano.constants import TIME_BETWEEN_PULSES, CHANNELS_PER_PULSE, channel_time
from euxfel_nano.io import read, read_ion, read_pnccd, data_root
from euxfel_nano.selection import (ion_selection, select_right_of_line,
                                   fish_plot_ion_selection)
from euxfel_nano.calibration import apply_calibration, mq_of, tof_of
from euxfel_nano.fields import build_charge_map, build_charge_maps
from euxfel_nano.yields import (FieldBinnedData, get_yields, significance,
                                verify_errors, sat_func, dec_func,
                                hist_for_selection)

P = config.params()
print("data root:", data_root())

## Fig. S10 — insensitivity to the ion-count windowThe analysis is repeated for each of the six windows in `config/params.yaml`.In the original notebook each window was a hand-copied block, and three of thesix were missing; here the window is the loop variable.

In [ ]:
divider = P["spatial"]["divider_line"]
THRESHOLD = [(500, 5000)]
DIAMETERS = config.measured_diameters_m()
NC = {"500": "selected_etof500.nc", "300": "selected_etof300.nc",
      "100": "selected_etof100.nc", "cs": "selected_etof_coreshell.nc"}

# Read once; the window only changes which pulses feed the charge maps.
base = {}
for key, nc_name in NC.items():
    dfevent, dfpulse = ion_selection(config.run_ids(key), THRESHOLD)[0]
    etof = xr.open_dataarray(nc_name)
    base[key] = select_right_of_line(dfevent, dfpulse, etof, divider[0], divider[1])


def data_for_window(nevents):
    """FieldBinnedData for one per-shot ion-count window."""
    event_dfs, charge_maps = {}, {}
    for key, (dfevent, dfpulse, etof) in base.items():
        bounded_dfpulse = dfpulse[(dfpulse.nevents_pulse > nevents[0])
                                  & (dfpulse.nevents_pulse < nevents[1])]
        bounded = dfevent[dfevent.trainId.isin(bounded_dfpulse.trainId)]
        event_dfs[key] = dfevent
        charge_maps[key] = build_charge_map(etof, bounded, DIAMETERS[key])
    return FieldBinnedData(event_dfs=event_dfs, charge_maps=charge_maps,
                           charge_windows=config.field_bins(),
                           tof_range=tuple(P["analysis_tof_range_s"]))

In [ ]:
windows = P["ion_count_windows"]["supplementary_scan"]
integration_ranges = config.species_windows_ns()
styles = {"H+": ("#1f77b4", "o", "--"), "H2+": ("#d62728", "s", ":"),
          "H3+": ("#2ca02c", "^", "-.")}

fig, axes = plt.subplots(3, 2, figsize=(12, 13), sharex=True, sharey=True)
for ax, (panel, nevents) in zip(axes.ravel(), sorted(windows.items())):
    d = data_for_window(tuple(nevents))
    E = np.linspace(d.field_centers.min(), d.field_centers.max(), 300)
    for species, span in integration_ranges.items():
        colour, marker, ls = styles[species]
        raw, err_raw, err_low = get_yields(d, span)
        model = dec_func if species == "H2+" else sat_func
        popt, _ = curve_fit(model, d.field_centers, raw,
                            sigma=np.clip(err_raw, 1e-6, None), absolute_sigma=True,
                            p0=[raw.max() - raw.min(), 1.5, raw.min()], maxfev=10000)
        ref = raw[0]
        ax.errorbar(d.field_centers, raw / ref, yerr=err_low, fmt=marker,
                    color=colour, ms=6, capsize=2, label=species)
        ax.plot(E, model(E, *popt) / ref, color=colour, ls=ls)
    ax.set_title(f"({panel}) {nevents[0]}-{nevents[1]} ion events", fontsize=12)
    ax.set_xlabel("Surface electric field (V/nm)")
    ax.set_ylabel("Norm. relative yield")
axes[0, 0].legend(fontsize=11)
plt.tight_layout(); plt.show()

## Fig. S11 — field dependence across the full spectrumThe Ar3+ gas-phase reference is field-insensitive while the H2O+ yield riseswith field; the silica-framework ions Si+, SiO+ and Si2+ show no monotonicdependence.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm

# ----------------------------------------------------------------------
# Shared configuration
# ----------------------------------------------------------------------
# TOF calibration (Cesar eq. 2, voltage setting 1):  tof = t0 + k*sqrt(m/q)
T0 = 7.45113e-9
K  = 2.06479e-7

def tof_of(mq):
    """Flight time (s) for a given m/q."""
    return T0 + K * np.sqrt(mq)

# Species markers, used for the zoom windows and for labelling
SPECIES = {
    "H+":     1,
    "H2+":    2,
    "H3+":    3,
    "Ar3+":   39.95 / 3,
    "H2O+":   18,
    "Si+":    28,
    "SiO+":   44,
    "Si2+":   56,     # di-silicon, NOT Si(2+)
    "Si2O+":  72,
}

charge_windows = [
    (0.2, 0.5),
    (0.5, 1),
    (1, 1.5),
    (1.5, 2),
    (2, 3),
    (3, 6),
    (10, 30),
]
field_colors = cm.viridis(np.linspace(0.05, 0.95, len(charge_windows)))

# ONE binning for every field window and for both figures.
# 1 ns bins over the full ion record: fine enough to resolve every peak,
# identical for all curves so amplitudes are directly comparable.
TOF_FULL   = (0.0, 3.3e-6)
BIN_WIDTH  = 1e-9
BINS_FULL  = np.arange(TOF_FULL[0], TOF_FULL[1] + BIN_WIDTH, BIN_WIDTH)
CENTERS    = 0.5 * (BINS_FULL[:-1] + BINS_FULL[1:])

# Reference window for normalisation. Fixed, identical for every curve and
# every figure, so the y-axis never depends on what the plot happens to show.
# Chosen to span the whole ion record.
NORM_RANGE = (0.15e-6, 3.0e-6)


# ----------------------------------------------------------------------
# Build one histogram per field window, pooled over particle types
# ----------------------------------------------------------------------
def build_field_histograms(event_dfs, charge_maps, bins=BINS_FULL,
                           normalise="per_shot", norm_range=NORM_RANGE,
                           ar3_halfwidth=12e-9):
    """
    Returns dict: (low, high) -> y array on `bins`.

    normalise:
      'per_shot'  -> counts per selected shot. Absolute, window-independent.
                     Use this to show that yields genuinely grow with field.
      'total'     -> divided by total counts in norm_range. Compositional.
                     Window-independent because norm_range is fixed.
      'ar3'       -> divided by the Ar3+ peak area. Field-referenced: Ar is
                     gas phase and field-insensitive, so this removes the
                     fluence common mode and leaves field-driven change.
    """
    out, nshots = {}, {}
    for (low, high) in charge_windows:
        h = np.zeros(len(bins) - 1, dtype=float)
        n = 0
        for key in event_dfs:
            cmap = charge_maps[key]
            sel_ids = cmap.index[(cmap.values >= low) & (cmap.values < high)].values
            if len(sel_ids) == 0:
                continue
            df2 = event_dfs[key]
            df2 = df2[df2["pulseId"].isin(sel_ids)]
            hh, _ = np.histogram(df2["tof"].values, bins=bins)
            h += hh
            n += len(sel_ids)

        if h.sum() == 0 or n == 0:
            continue

        if normalise == "per_shot":
            y = h / n
        elif normalise == "total":
            m = (CENTERS >= norm_range[0]) & (CENTERS < norm_range[1])
            y = h / h[m].sum()
        elif normalise == "ar3":
            t = tof_of(SPECIES["Ar3+"])
            m = (CENTERS >= t - ar3_halfwidth) & (CENTERS <= t + ar3_halfwidth)
            y = h / h[m].sum()
        else:
            raise ValueError(normalise)

        out[(low, high)] = y
        nshots[(low, high)] = n
    return out, nshots


# ----------------------------------------------------------------------
# Peak area vs field, with bootstrap errors.
# This is the panel that answers "they all look the same" with numbers.
# ----------------------------------------------------------------------
def peak_area(y, mq, halfwidth=12e-9, baseline_gap=20e-9, baseline_width=15e-9):
    """Integrate a peak above a local linear baseline taken on both sides."""
    t = tof_of(mq)
    peak = (CENTERS >= t - halfwidth) & (CENTERS <= t + halfwidth)
    lo = (CENTERS >= t - halfwidth - baseline_gap - baseline_width) & \
         (CENTERS <  t - halfwidth - baseline_gap)
    hi = (CENTERS >  t + halfwidth + baseline_gap) & \
         (CENTERS <= t + halfwidth + baseline_gap + baseline_width)
    if lo.sum() == 0 or hi.sum() == 0:
        return np.nan
    b = 0.5 * (y[lo].mean() + y[hi].mean())
    return (y[peak] - b).sum()


def area_vs_field(hists, species=("Ar3+", "H2O+", "Si+"), n_boot=200):
    """Peak area per field bin, with bootstrap errors from Poisson resampling."""
    rows = []
    for (low, high), y in hists.items():
        E = 0.5 * (low + high)
        row = {"E": E, "E_lo": low, "E_hi": high}
        for s in species:
            row[s] = peak_area(y, SPECIES[s])
            boots = [peak_area(np.random.poisson(np.clip(y, 0, None) * 1e4) / 1e4,
                               SPECIES[s]) for _ in range(n_boot)]
            row[s + "_err"] = np.nanstd(boots)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("E")


# ----------------------------------------------------------------------
# figS11 replacement: multi-panel with per-species zooms + baseline check
# ----------------------------------------------------------------------
def plot_figS11(hists, normalise_label="counts per shot",
                out="figS11_new.pdf"):
    fig = plt.figure(figsize=(5.147, 5.4))
    gs = fig.add_gridspec(3, 3, hspace=0.55, wspace=0.35,
                          height_ratios=[1.0, 1.0, 1.0])

    # (a) full spectrum
    ax = fig.add_subplot(gs[0, :])
    for (win, y), c in zip(hists.items(), field_colors):
        ax.plot(CENTERS * 1e6, y, lw=1.0, color=c,
                label=f"{win[0]:g}\u2013{win[1]:g} V/nm")
    ax.set_xlim(0, 3.0)
    ax.set_yscale("log")
    ax.set_xlabel("Time of flight (\u03bcs)", fontsize=7)
    ax.set_ylabel(normalise_label, fontsize=7)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=5.5, frameon=False, ncol=2, loc="upper right")
    ax.text(-0.06, 1.06, "a", transform=ax.transAxes,
            fontsize=8, fontweight="bold")

    # (b) pre-ion baseline: nothing can arrive before H+ at ~214 ns.
    # If the curves are separated here, the offset is instrumental.
    ax = fig.add_subplot(gs[1, 0])
    for (win, y), c in zip(hists.items(), field_colors):
        ax.plot(CENTERS * 1e9, y, lw=1.0, color=c)
    ax.axvline(tof_of(1) * 1e9, ls=":", lw=0.8, color="k")
    ax.set_xlim(0, 220)
    ax.set_xlabel("TOF (ns)", fontsize=7)
    ax.set_ylabel(normalise_label, fontsize=7)
    ax.tick_params(labelsize=7)
    ax.set_title("pre-ion baseline", fontsize=6)
    ax.text(-0.20, 1.10, "b", transform=ax.transAxes,
            fontsize=8, fontweight="bold")

    # (c)-(f) per-species zooms, all on identical bins
    zooms = [("Ar3+", 60), ("H2O+", 60), ("Si+", 60), ("Si2+", 60)]
    slots = [gs[1, 1], gs[1, 2], gs[2, 0], gs[2, 1]]
    letters = ["c", "d", "e", "f"]
    for (name, half_ns), slot, L in zip(zooms, slots, letters):
        ax = fig.add_subplot(slot)
        t0 = tof_of(SPECIES[name]) * 1e9
        for (win, y), c in zip(hists.items(), field_colors):
            ax.plot(CENTERS * 1e9, y, lw=1.0, color=c)
        ax.set_xlim(t0 - half_ns, t0 + half_ns)
        ax.set_xlabel("TOF (ns)", fontsize=7)
        ax.tick_params(labelsize=7)
        ax.set_title(name, fontsize=6)
        ax.text(-0.20, 1.10, L, transform=ax.transAxes,
                fontsize=8, fontweight="bold")

    # (g) hydrogen region, the one ER asks about
    ax = fig.add_subplot(gs[2, 2])
    for (win, y), c in zip(hists.items(), field_colors):
        ax.plot(CENTERS * 1e9, y, lw=1.0, color=c)
    for s in ("H+", "H2+", "H3+"):
        ax.axvline(tof_of(SPECIES[s]) * 1e9, ls=":", lw=0.6, color="k")
    ax.set_xlim(180, 420)
    ax.set_xlabel("TOF (ns)", fontsize=7)
    ax.tick_params(labelsize=7)
    ax.set_title("H$^+$ / H$_2^+$ / H$_3^+$", fontsize=6)
    ax.text(-0.20, 1.10, "g", transform=ax.transAxes,
            fontsize=8, fontweight="bold")

    fig.savefig(out, bbox_inches="tight")
    return fig


# ----------------------------------------------------------------------
# Quantitative panel: peak area vs field
# ----------------------------------------------------------------------
def plot_area_vs_field(df, out="figS11_areas.pdf"):
    fig, ax = plt.subplots(figsize=(5.147, 2.2))
    for s, m in (("Ar3+", "o"), ("H2O+", "s"), ("Si+", "^")):
        ax.errorbar(df["E"], df[s] / df[s].iloc[0],
                    yerr=df[s + "_err"] / df[s].iloc[0],
                    marker=m, ms=3, lw=1.0, capsize=2, label=s)
    ax.axhline(1.0, ls=":", lw=0.8, color="k")
    ax.set_xscale("log")
    ax.set_xlabel("Surface electric field (V/nm)", fontsize=7)
    ax.set_ylabel("Peak area (normalised to\nlowest field bin)", fontsize=7)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6, frameon=False)
    fig.savefig(out, bbox_inches="tight")
    return fig


# ----------------------------------------------------------------------
# Usage
# ----------------------------------------------------------------------
if __name__ == "__main__":
    event_dfs = {
        "500": ion_dfevent500,          # noqa: F821
        "300": ion_dfevent300,          # noqa: F821
        "100": ion_dfevent100,          # noqa: F821
        "cs":  coreshell_ion_dfevent,   # noqa: F821
    }
    charge_maps = {
        "500": charge_map500,           # noqa: F821
        "300": charge_map300,           # noqa: F821
        "100": charge_map100,           # noqa: F821
        "cs":  charge_map_cs,           # noqa: F821
    }

    # Per-shot first: this is the honest absolute picture and needs no
    # window-dependent normalisation at all.
    hists, nshots = build_field_histograms(event_dfs, charge_maps,
                                           normalise="per_shot")
    print("shots per field bin:", nshots)
    plot_figS11(hists, "Ions per shot")

    # Ar3+-referenced: removes the fluence common mode, leaves field response.
    hists_ar, _ = build_field_histograms(event_dfs, charge_maps,
                                         normalise="ar3")
    plot_figS11(hists_ar, "Yield / Ar$^{3+}$ area", out="figS11_ar3ref.pdf")

    # The panel that ends the argument.
    df = area_vs_field(hists)
    print(df)
    plot_area_vs_field(df)

In [ ]:
# =====================================================================
# figS11 + zooms.  Original tof_range and bins_per_window restored.
# Curves are built once and reused in the main panel and every zoom.
# =====================================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

tof_range = (1e-7, 2e-6)
bins_per_window = [500, 250, 250, 250, 150, 80, 100]
field_colors = cm.viridis(np.linspace(0.05, 0.95, len(charge_windows)))  # noqa: F821

peak_windows = {
    "Ar$^{3+}$":   (880e-9,  970e-9),
    "H$_2$O$^+$":  (1000e-9, 1100e-9),
    "Si$^+$":      (1.10e-6, 1.30e-6),
    "SiO$^+$":     (1.47e-6, 1.55e-6),
    "Si$_2^+$":    (1.60e-6, 1.70e-6),
}


def hist_for_selection(df, pulse_ids, bins, tof_range):
    df2 = df[df["pulseId"].isin(pulse_ids)]
    df2 = df2[df2["tof"].between(tof_range[0], tof_range[1])]
    h, _ = np.histogram(df2["tof"].values, bins=bins)
    return h


# each field window keeps its own binning -> store its own x axis
curves = {}
for (low, high), NBINS in zip(charge_windows, bins_per_window):   # noqa: F821
    bins = np.linspace(tof_range[0], tof_range[1], NBINS + 1)
    total_hist = np.zeros(NBINS, dtype=float)
    for key in ("500", "300", "100", "cs"):
        cmap = charge_maps[key]                                   # noqa: F821
        sel_ids = cmap.index[(cmap.values >= low) & (cmap.values < high)].values
        total_hist += hist_for_selection(event_dfs[key], sel_ids, bins, tof_range)  # noqa: F821
    if total_hist.sum() > 0:
        curves[(low, high)] = (bins[:-1], total_hist * NBINS / total_hist.sum())

# ---------------------------------------------------------------------
fig = plt.figure(figsize=(13, 8))
gs = fig.add_gridspec(2, len(peak_windows), hspace=0.35, wspace=0.3,
                      height_ratios=[1.3, 1.0])

# (a) same view as before, with the zoom windows marked
ax = fig.add_subplot(gs[0, :])
for ((low, high), (x, y)), c in zip(curves.items(), field_colors):
    ax.plot(x * 1e9, y, lw=2.2, color=c, label=f"{low:g}\u2013{high:g} V/nm")
for name, (t0, t1) in peak_windows.items():
    ax.axvspan(t0 * 1e9, t1 * 1e9, color="0.88", zorder=0)
    ax.text(0.5 * (t0 + t1) * 1e9, 1.88, name, ha="center", fontsize=11)
ax.set_xlim(800, 1750)
ax.set_ylim(0, 2)
ax.set_xlabel("Time of flight (ns)", fontsize=18)
ax.set_ylabel("Relative ion yields (arb. unit)", fontsize=18)
ax.tick_params(labelsize=14)
ax.legend(fontsize=12, ncol=2)

# (b..f) zooms, auto y so each peak fills its panel
for j, (name, (t0, t1)) in enumerate(peak_windows.items()):
    ax = fig.add_subplot(gs[1, j])
    ymax = 0
    for ((low, high), (x, y)), c in zip(curves.items(), field_colors):
        m = (x >= t0) & (x <= t1)
        ax.plot(x[m] * 1e9, y[m], lw=2.0, color=c)
        if m.any():
            ymax = max(ymax, y[m].max())
    ax.set_xlim(t0 * 1e9, t1 * 1e9)
    ax.set_ylim(0, ymax * 1.15)
    ax.set_title(name, fontsize=13)
    ax.set_xlabel("TOF (ns)", fontsize=12)
    if j == 0:
        ax.set_ylabel("Relative ion yields", fontsize=12)
    ax.tick_params(labelsize=11)

plt.tight_layout()
plt.show()


# =====================================================================
# eTOF: does the 100-200 ns window capture a CONSTANT fraction?
# Flat vs ion count -> window only rescales E, absolute scale is fixed
# by the SIMION cross-check, and the question is closed with a sentence.
# Sloping -> the field axis is distorted in shape, not just scale.
# =====================================================================
etof = selected_etof300                                  # noqa: F821
ic_all = bounded_dfevent300.pulseId.value_counts()       # noqa: F821

# the fix: only pulseIds present in BOTH
ids = np.intersect1d(ic_all.index.values, etof.pulseId.values)
ic = ic_all.loc[ids]
sub = etof.sel(pulseId=ids)

c_lo  = int(1e-7 / channel_time)                         # noqa: F821
c_200 = int(2e-7 / channel_time)                         # noqa: F821
N_200 = -np.sum(sub.isel(data=slice(c_lo, c_200)).values, axis=1)

# drop shots where N_200 is too small for a ratio to mean anything
good = N_200 > np.percentile(N_200, 20)

plt.figure(figsize=(9, 4))
for upper, col in zip((2.5e-7, 3e-7, 4e-7), ("tab:blue", "tab:orange", "tab:green")):
    c_hi = int(upper / channel_time)                     # noqa: F821
    N = -np.sum(sub.isel(data=slice(c_lo, c_hi)).values, axis=1)
    ratio = N[good] / N_200[good]
    x = ic.values[good]
    plt.scatter(x, ratio, s=2, alpha=0.12, color=col, edgecolors="none")
    # binned median so the trend is readable through the scatter
    edges = np.quantile(x, np.linspace(0, 1, 12))
    cx, cy = [], []
    for i in range(len(edges) - 1):
        m = (x >= edges[i]) & (x < edges[i + 1])
        if m.sum() > 10:
            cx.append(np.median(x[m])); cy.append(np.median(ratio[m]))
    plt.plot(cx, cy, lw=2.0, color=col, label=f"100\u2013{upper*1e9:.0f} ns")

plt.axhline(1.0, ls=":", lw=1.0, color="k")
plt.xlabel("Ion count per shot", fontsize=14)
plt.ylabel(r"$N$(wider) / $N$(100\u2013200 ns)", fontsize=14)
plt.tick_params(labelsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()